# Weight Initialization

> Based on Stanford CS230 — Lecture Notes 3, Andrew Ng / DeepLearning.AI

How you initialise $W^{[l]}$ has a dramatic effect on training speed and convergence. The wrong initialisation can cause vanishing/exploding gradients before training even begins.

## Learning Objectives

1. Explain why zero initialisation fails (symmetry problem)
2. Implement random, He, and Xavier initialisation
3. Observe the effect of each strategy on activation distributions
4. Choose the right initialisation for a given activation function

## Why Not Zero?

If $W^{[l]} = 0$, then $Z^{[l]} = 0$ for all units in every layer. Every neuron in a layer computes the **same function**, and gradient descent updates them identically — they never differentiate. This is the **symmetry breaking problem**.

Biases $b^{[l]} = 0$ is fine; only weights must break symmetry.

## Initialisation Strategies

| Method | Formula | Recommended for |
|---|---|---|
| **Random** | $W \sim \mathcal{N}(0, 0.01^2)$ | Small networks; often too small |
| **Xavier / Glorot** | $W \sim \mathcal{N}\!\left(0,\; \tfrac{1}{n^{[l-1]}}\right)$ | Tanh, sigmoid activations |
| **He** | $W \sim \mathcal{N}\!\left(0,\; \tfrac{2}{n^{[l-1]}}\right)$ | ReLU activations |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

def forward_acts(X, dims, init_fn, activation=relu):
    """Return list of activation matrices for each layer."""
    acts = [X]
    A = X
    np.random.seed(0)
    for i in range(1, len(dims)):
        W = init_fn(dims[i], dims[i-1])
        Z = W @ A
        A = activation(Z)
        acts.append(A)
    return acts

def init_zeros(n_out, n_in):     return np.zeros((n_out, n_in))
def init_random(n_out, n_in):    return np.random.randn(n_out, n_in) * 0.01
def init_xavier(n_out, n_in):    return np.random.randn(n_out, n_in) * np.sqrt(1 / n_in)
def init_he(n_out, n_in):        return np.random.randn(n_out, n_in) * np.sqrt(2 / n_in)
def init_large(n_out, n_in):     return np.random.randn(n_out, n_in) * 10.0  # pathological

inits = {
    'Zero':          (init_zeros,  relu),
    'Too small (×0.01)': (init_random, relu),
    'Xavier':        (init_xavier, sigmoid),
    'He (ReLU)':     (init_he,     relu),
    'Too large (×10)': (init_large, sigmoid),
}

np.random.seed(1)
dims = [500, 200, 100, 50, 20, 10]  # 5 hidden layers
X = np.random.randn(dims[0], 1000)

fig, axes = plt.subplots(len(inits), len(dims)-1, figsize=(16, 14))
fig.suptitle('Activation Distributions After Each Layer\n(row = init strategy)', fontsize=13, fontweight='bold')

for row, (name, (init_fn, act_fn)) in enumerate(inits.items()):
    acts = forward_acts(X, dims, init_fn, act_fn)
    for col in range(1, len(dims)):
        ax = axes[row][col-1]
        data = acts[col].flatten()
        ax.hist(data, bins=30, color=P[row % len(P)], alpha=0.8, edgecolor='none')
        std = np.std(data)
        ax.set_title(f'Layer {col}  σ={std:.2e}', fontsize=8)
        ax.set_yticks([])
        ax.tick_params(labelsize=7)
        if col == 1:
            ax.set_ylabel(name, fontsize=9, rotation=0, ha='right', labelpad=60)
        ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# --- Training comparison: zero vs He ---
def sigmoid_d(z): s = sigmoid(z); return s*(1-s)

def train_init(X, Y, dims, init_fn, activation=relu, deriv=None, lr=0.05, n=2000):
    np.random.seed(42)
    L = len(dims) - 1
    params = {}
    for l in range(1, L+1):
        params[f'W{l}'] = init_fn(dims[l], dims[l-1])
        params[f'b{l}'] = np.zeros((dims[l], 1))
    costs = []
    m = Y.shape[1]
    for _ in range(n):
        # forward
        A, cache = X, []
        for l in range(1, L+1):
            W, b = params[f'W{l}'], params[f'b{l}']
            Z = W @ A + b
            A_prev = A
            A = relu(Z) if l < L else sigmoid(Z)
            cache.append((A_prev, W, b, Z, A))
        # cost
        costs.append(-np.mean(Y*np.log(A+1e-8)+(1-Y)*np.log(1-A+1e-8)))
        # backward (output)
        dA = -(Y/(A+1e-8) - (1-Y)/(1-A+1e-8))
        A_prev_l, Wl, bl, Zl, Al = cache[-1]
        dZ = dA * Al*(1-Al)
        A_prev_idx = cache[-2][4] if L > 1 else cache[0][0]
        params[f'W{L}'] -= lr * (dZ @ A_prev_idx.T) / m
        params[f'b{L}'] -= lr * np.mean(dZ, axis=1, keepdims=True)
        dA = params[f'W{L}'].T @ dZ
        for l in reversed(range(1, L)):
            A_prev_l, Wl, bl, Zl, Al = cache[l-1]
            dZ = dA * (Zl > 0).astype(float)
            params[f'W{l}'] -= lr * (dZ @ A_prev_l.T) / m
            params[f'b{l}'] -= lr * np.mean(dZ, axis=1, keepdims=True)
            dA = params[f'W{l}'].T @ dZ
    return costs

np.random.seed(0)
m = 300
X_tr = np.vstack([np.random.randn(2, m//2)+[[2],[2]],
                   np.random.randn(2, m//2)+[[-2],[-2]]]).T.T
Y_tr = np.array([1]*(m//2)+[0]*(m//2)).reshape(1,-1)
dims_tr = [2, 10, 7, 1]

strategies = [
    ('He init',          init_he,    P[3]),
    ('Xavier init',      init_xavier, P[0]),
    ('Too small (0.01)', init_random,  P[2]),
    ('Too large (×10)',  init_large,   P[1]),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('Training Cost by Initialisation Strategy', fontsize=12)
for label, fn, color in strategies:
    costs = train_init(X_tr, Y_tr, dims_tr, fn)
    ax.plot(costs, color=color, lw=2, label=f'{label}  (final={costs[-1]:.4f})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost J')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Intuition — Why He Works for ReLU

He initialisation sets variance $= 2/n^{[l-1]}$ so that the **variance of $Z^{[l]}$ is approximately 1** after a ReLU layer (which zeros out ~half the inputs, effectively halving the variance):

$$\text{Var}(Z^{[l]}) = n^{[l-1]} \cdot \text{Var}(W^{[l]}) \cdot \text{Var}(A^{[l-1]})$$

With ReLU, $\text{Var}(A^{[l-1]}) \approx \tfrac{1}{2}\text{Var}(Z^{[l-1]})$.  
Setting $\text{Var}(W^{[l]}) = \tfrac{2}{n^{[l-1]}}$ keeps the variance stable across layers.

Xavier uses $\tfrac{1}{n^{[l-1]}}$ — designed for tanh/sigmoid whose full output range is used symmetrically around zero.

> **Rule of thumb**: He for ReLU, Xavier for tanh/sigmoid. Never zero, never too large.
